In [ ]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


In [ ]:
df = pd.read_csv("heart.csv")

In [ ]:
df.head()

# EDA

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.isnull().sum()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.duplicated().sum()

In [ ]:
df['HeartDisease'].value_counts()

In [ ]:
df['HeartDisease'].value_counts().plot(kind="bar")

In [ ]:
def plotting(var , num): 
    plt.subplot(2,2,num)
    sns.histplot(df[var],kde=True)

plotting("Age",1)
plotting("RestingBP",2)
plotting("Cholesterol",3)
plotting("MaxHR",4)

plt.tight_layout()

In [ ]:
df['Cholesterol'].value_counts()

In [ ]:
df.loc[df['Cholesterol']==0 ,"Cholesterol"]=np.nan

In [ ]:
df['Cholesterol'] = df['Cholesterol'].fillna(df["Cholesterol"].mean())

In [ ]:
df['Cholesterol']=df['Cholesterol'].round(2)

In [ ]:
df["RestingBP"]=df['RestingBP'].replace(0,np.nan)
df["RestingBP"]=df["RestingBP"].fillna(df['RestingBP'].mean())
df["RestingBP"]=df['RestingBP'].round(2)

In [ ]:
def plotting(var , num): 
    plt.subplot(2,2,num)
    sns.histplot(df[var],kde=True)

plotting("Age",1)
plotting("RestingBP",2)
plotting("Cholesterol",3)
plotting("MaxHR",4)

plt.tight_layout()

In [ ]:
sns.countplot(x=df['Sex'],hue=df['HeartDisease'])

In [ ]:
sns.countplot(x=df['ChestPainType'],hue=df['HeartDisease'])

In [ ]:
sns.countplot(x=df['FastingBS'],hue=df['HeartDisease'])

In [ ]:
sns.boxplot(x=df['HeartDisease'],y= df['Cholesterol'])

In [ ]:
 sns.violinplot(x=df['HeartDisease'],y=df['Age'])

In [ ]:
sns.heatmap(df.corr(numeric_only=True),annot=True)

# Data preprocessing and Cleaning 

In [ ]:
 df_encoded = pd.get_dummies(df,drop_first=True)

In [ ]:
df_encoded

In [ ]:
df_encoded = df_encoded.astype(int)

In [ ]:
df_encoded

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
numerical_cols = ['Age','RestingBP','Cholesterol','MaxHR','Oldpeak']

In [ ]:
scaler = StandardScaler()

df_encoded[numerical_cols] = scaler.fit_transform(df_encoded[numerical_cols])

In [ ]:
df_encoded.head()

# Feature Extraction and Engineering

In [ ]:
df_encoded.columns

In [ ]:
df_encoded['IsOld'] =  df_encoded['Age']>0

In [ ]:
df_encoded

In [ ]:
df_encoded['Age']=df_encoded['Age'].astype(int)
df_encoded

In [ ]:
df_encoded['HeartWork'] = df_encoded['RestingBP'] + df_encoded['MaxHR']

In [ ]:
df_encoded.head()

In [ ]:
df_encoded['CholesterolLevel'] = pd.qcut(
    df_encoded['Cholesterol'],
    q=3,
    labels=['Low', 'Normal', 'High']
)

In [ ]:
df_encoded.head()

In [ ]:
df_encoded = pd.get_dummies(df_encoded, columns=['CholesterolLevel'])

In [ ]:
df_encoded.head()

In [ ]:
df_encoded.columns

In [ ]:
df_encoded = df_encoded.loc[:, ~df_encoded.columns.str.startswith('Cholesterol_')]

In [ ]:
df_encoded = df_encoded.drop('Cholesterol', axis=1, errors='ignore')

In [ ]:
df_encoded.head()


In [ ]:
df_encoded.columns

In [ ]:
df_encoded.head()

In [ ]:
bool_cols = ['IsOld', 'CholesterolLevel_Low', 'CholesterolLevel_Normal', 'CholesterolLevel_High']
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)


In [ ]:
selected_features = ['Age', 'RestingBP', 'FastingBS', 'MaxHR', 'Oldpeak',
                     'Sex_M', 'ChestPainType_ATA', 'ChestPainType_NAP', 'ChestPainType_TA',
                     'RestingECG_Normal', 'RestingECG_ST', 'ExerciseAngina_Y',
                     'ST_Slope_Flat', 'ST_Slope_Up', 'IsOld', 'HeartWork',
                     'CholesterolLevel_Low', 'CholesterolLevel_Normal', 'CholesterolLevel_High']


In [ ]:

from scipy.stats import pearsonr
correlations = {
    feature: pearsonr(df_encoded[feature], df_encoded['HeartDisease'])[0]
    for feature in selected_features
}

corr_df = pd.DataFrame(list(correlations.items()),
                       columns=['feature', 'correlation'])
corr_df = corr_df.sort_values(by='correlation', ascending=False)

In [ ]:
corr_df

In [ ]:
from scipy.stats import chi2_contingency

In [ ]:
categorical_features = [
    'Sex_M', 'ChestPainType_ATA', 'ChestPainType_NAP', 'ChestPainType_TA',
    'RestingECG_Normal', 'RestingECG_ST', 'ExerciseAngina_Y',
    'ST_Slope_Flat', 'ST_Slope_Up', 'IsOld',
    'CholesterolLevel_Low', 'CholesterolLevel_Normal', 'CholesterolLevel_High',
    'FastingBS'
]

In [ ]:
alpha = 0.05

results = []
for feature in categorical_features:

    # contingency table
    table = pd.crosstab(df_encoded[feature], df_encoded['HeartDisease'])

    # chi-square test
    chi2, p, dof, expected = chi2_contingency(table)

    # decision rule
    if p < alpha:
        decision = "Reject H0 (Significant relationship)"
    else:
        decision = "Fail to reject H0 (No significant relationship)"

    results.append([feature, chi2, p, decision])


In [ ]:
chi_df = pd.DataFrame(
    results,
    columns=['feature', 'chi2_stat', 'p_value', 'decision']
)


In [ ]:
chi_df = chi_df.sort_values(by='p_value')


In [ ]:
chi_df

In [ ]:
selected_features = [

    'ST_Slope_Up',      
    'ST_Slope_Flat',      
    'ExerciseAngina_Y',  
    'ChestPainType_ATA',  
    'MaxHR',             
    'Oldpeak',            
    'Sex_M',            
    'FastingBS',        
    'IsOld',             
    'Age',                
    'ChestPainType_NAP',  
    'HeartWork',          
]

In [ ]:
final_df = df_encoded[selected_features]

In [ ]:
final_df